# 07 · Audit vs Ground Truth  ·  the ONLY pipeline reader of ground truth
Joins pipeline outputs to the sealed manifest and reports honestly, misses included: entity counts + mapping, mention recall/precision with itemized misses, B-cubed cluster quality with over/under-merge evidence trails, the scan-coverage proof, and hash re-verification.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
from src.repository import Repository
from src import audit
repo = Repository()
report = audit.run(repo)
print(report['summary'])


In [ ]:
m = report['entity_mapping']
print('GT entities            :', m['gt_entity_count'])
print('system clusters        :', m['system_entity_count'])
print('GT never recovered     :', m['n_gt_never_recovered'])


In [ ]:
r = report['mention_recall']; p = report['mention_precision']
print('mention recall  :', r['recall'], '(', r['found'], '/', r['total_placements'], ')')
print('mention precision:', p['precision'], '| planted non-entities wrongly extracted:', p['fp_nonentity_planted'])
print('recall by segment_kind:'); 
import json; print(json.dumps(r['by_segment_kind'], indent=1))
print('recall by hard case:'); print(json.dumps(r['by_hard_case'], indent=1))
print('sample misses (doc_id, span, variant):')
for miss in r['missed_sample'][:8]: print('  ', miss['doc_id'], miss['span'], repr(miss['surface_variant']), miss['segment_kind'])


In [ ]:
c = report['cluster_quality']
print('B-cubed  precision/recall/F1:', c['bcubed_precision'], c['bcubed_recall'], c['bcubed_f1'])
print('over-merges:', c['n_over_merges'], '| under-merges:', c['n_under_merges'])
print('over-merge evidence trail (sample):')
import json
for om in report['over_merge_evidence'][:2]: print(json.dumps(om, indent=1)[:700])


In [ ]:
cov = report['coverage_proof']
print('SCAN-COVERAGE PROOF')
print('  overall coverage     :', cov['overall_coverage'])
print('  docs at 100%%         :', cov['n_docs_full_coverage'], '/', cov['n_docs'])
print('  coverage histogram   :', cov['coverage_histogram'])
print('  overlap depth (chars):', cov['overlap_depth_chars'], '(fraction', cov['overlap_fraction'], ')')
print('  docs under 100%%      :', cov['n_docs_under_100pct'])
print('hash re-verification  :', report['hash_verification']['ok'])
repo.close()
